In [1]:
#@markdown General Preamble (loads necessary libraries)
import os, time, random, requests, datetime, urllib
from bs4 import BeautifulSoup, NavigableString, Tag

import pandas as pd
from urllib.parse import urljoin


In [3]:
#@markdown URLs (found via sitemap)
urls = [
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-17-mars-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-18-juillet-2007.html ",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-19-juillet-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/albanie-18-janvier-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/albanie-22-juillet-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/albanie-20-decembre-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/algerie-21-juillet-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/algerie-1er-juin-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/angola-31-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/angola-20-juillet-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/antigua-et-barbuda-16-septembre-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-29-mai-2014.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-22-juillet-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-19-septembre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-21-decembre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-20-mai-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-16-janvier-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-26-juin-1965.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-24-octobre-1962.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/argentine-16-mai-1956.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/benin-23-avril-2003.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/benin-24-octobre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/benin-24-octobre-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/benin-21-juin-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/benin-18-decembre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/benin-22-juin-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-10-juillet-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-30-octobre-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-15-decembre-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-24-mars-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-24-janvier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-15-mars-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-14-novembre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bolivie-18-juillet-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bosnie-herzegovine-12-juillet-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bosnie-herzegovine-28-octobre-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bresil-26-fevrier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bresil-29-juillet-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bresil-21-janvier-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bresil-23-novembre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bresil-1er-juillet-1964.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bresil-24-mai-1961.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bulgarie-13-avril-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bulgarie-14-decembre-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/bulgarie-17-avril-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burkina-faso-26-mai-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burkina-faso-20-juin-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burkina-faso-24-octobre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burkina-faso-20-juin-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burkina-faso-7-mai-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burkina-faso-15-mars-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burundi-11-mars-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burundi-15-septembre-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/burundi-4-mars-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cambodge-26-janvier-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cambodge-31-octobre-1972.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cambodge-27-janvier-1972.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-19-mai-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-17-juin-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-24-janvier-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-24-octobre-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-16-novembre-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-25-mars-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-23-janvier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cameroun-24-mai-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cap-vert-12-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/chili-2-avril-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/chili-17-juillet-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/chili-25-mars-1974.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/chili-19-avril-1972.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/chili-24-fevrier-1965.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maldives-14-septembre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/comores-28-fevrier-2013.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/comores-13-aout-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/comores-19-novembre-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-9-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-18-mars-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-11-decembre-2008.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-9-mars-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-16-decembre-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-16-juillet-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-30-juin-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-13-septembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/congo-18-juillet-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/costa-rica-22-juin-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/costa-rica-16-juillet-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/costa-rica-26-mai-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/costa-rica-22-avril-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/costa-rica-11-janvier-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-11-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-29-juin-2012.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-15-novembre-2011.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-15-mai-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-10-avril-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-24-avril-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-23-mars-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-20-novembre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-18-decembre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-18-decembre-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-27-juin-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-25-juin-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/cote-d-ivoire-4-mai-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/croatie-21-mars-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/djibouti-10-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/djibouti-16-octobre-2008.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/djibouti-25-mai-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/dominique-15-mai-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/egypte-25-mai-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/egypte-22-mai-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/el-salvador-17-septembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-13-juin-2003.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-15-septembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-27-juin-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-20-janvier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-24-octobre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-20-janvier-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-24-avril-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/equateur-28-juillet-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ethiopie-9-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ethiopie-13-mai-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ethiopie-18-avril-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ethiopie-5-avril-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ethiopie-24-janvier-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ethiopie-16-decembre-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ex-yougoslavie-13-juillet-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ex-yougoslavie-13-mai-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ex-yougoslavie-24-mai-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ex-yougoslavie-22-mai-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/federation-de-russie-1-aout-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/federation-de-russie-29-avril-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/federation-de-russie-3-juin-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/federation-de-russie-4-juin-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/federation-de-russie-2-avril-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/fidji-13-octobre-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-11-juin-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-15-decembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-12-decembre-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-15-avril-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-24-octobre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-19-septembre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-21-mars-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gabon-21-janvier-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gambie-24-janvier-2008.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gambie-22-juin-2007.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gambie-9-janvier-2003.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/gambie-19-septembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/georgie-21-juillet-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/georgie-6-mars-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ghana-11-juin-2024.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ghana-22-juillet-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ghana-16-mai-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ghana-10-decembre-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ghana-18-avril-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/grenade-18-mai-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/grenade-19-novembre-2015.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/grenade-12-mai-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guatemala-25-mars-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-24-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-25-octobre-2012.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-11-avril-2012.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-23-janvier-2008.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-15-mai-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-26-fevrier-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-25-janvier-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-18-novembre-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-12-avril-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-18-avril-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-equatoriale-15-decembre-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-equatoriale-2-avril-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-equatoriale-1-mars-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-equatoriale-22-juillet-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-26-avril-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-10-mai-2011.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-6-juillet-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-26-janvier-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-23-fevrier-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-26-octobre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guinee-bissau-27-octobre-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guyana-14-janvier-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guyana-25-juin-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guyana-23-mai-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guyana-6-mai-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guyana-12-septembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/guyana-24-mai-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/haiti-8-juillet-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/haiti-12-decembre-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/haiti-30-mai-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/honduras-12-mai-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/honduras-14-avril-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/honduras-13-avril-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/honduras-1-mars-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/honduras-26-octobre-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/honduras-14-septembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-10-mai-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-12-avril-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-13-avril-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-23-septembre-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-24-avril-1970.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-17-octobre-1968.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-18-octobre-1967.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/indonesie-20-decembre-1966.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/irak-21-novembre-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-25-janvier-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-19-juillet-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-26-avril-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-24-octobre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-5-mars-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-19-juillet-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jamaique-16-juillet-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jordanie-10-juillet-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jordanie-20-mai-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jordanie-23-mai-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jordanie-28-juin-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jordanie-28-fevrier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/jordanie-19-juillet-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kenya-11-janvier-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kenya-15-janvier-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kenya-15-novembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kenya-19-janvier-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kirghizie-10-septembre-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kirghizie-11-mars-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/kirghizie-7-mars-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/lesotho-9-septembre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/liberia-16-septembre-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/liberia-17-avril-2008.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/liberia-17-decembre-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/liberia-22-decembre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/liberia-16-decembre-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/liberia-19-decembre-1980.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/macedoine-ex-yougoslavie-11-septembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/macedoine-ex-yougoslavie-17-juillet-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-12-octobre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-16-novembre-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-7-mars-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-4-septembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-26-mars-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-10-juillet-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-28-octobre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-23-octobre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-22-mai-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-23-mars-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-13-juillet-1982.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/madagascar-30-avril-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/malawi-19-octobre-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/malawi-25-janvier-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/malawi-22-avril-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/malawi-27-octobre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/malawi-22-septembre-1982.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/comores-15-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-15-mai-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-12-mars-2003.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-25-octobre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-20-mai-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-29-octobre-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-22-novembre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mali-27-octobre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maroc-27-fevrier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maroc-11-septembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maroc-26-octobre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maroc-6-mars-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maroc-17-septembre-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/maroc-25-octobre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-2-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-8-juillet-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-16-mars-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-28-juin-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-26-janvier-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-19-juin-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-15-juin-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-16-mai-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mauritanie-27-avril-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mexique-30-mai-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mexique-17-septembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mexique-22-juin-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/moldavie-12-mai-2006.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/montenegro-16-novembre-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-29-septembre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-17-novembre-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-9-juillet-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-25-mai-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-21-novembre-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-23-mars-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-14-juin-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-16-juin-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/mozambique-25-octobre-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/myanmar-10-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/myanmar-25-janvier-2013.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nepal-19-mai-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nicaragua-4-mars-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nicaragua-13-decembre-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nicaragua-16-mars-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nicaragua-22-avril-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nicaragua-22-mars-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nicaragua-17-decembre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-4-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-12-mai-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-25-janvier-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-19-decembre-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-4-mars-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-18-septembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-16-decembre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-21-avril-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-20-novembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-21-novembre-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-30-novembre-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/niger-14-novembre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nigeria-20-octobre-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nigeria-13-decembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nigeria-18-janvier-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nigeria-3-mars-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/nigeria-16-decembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-28-janvier-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-12-septembre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-24-avril-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-20-fevrier-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-17-juin-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-26-janvier-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-19-juin-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-1-decembre-1982.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ouganda-18-novembre-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-9-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-13-decembre-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-23-janvier-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-30-janvier-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-14-janvier-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-28-juin-1974.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pakistan-26-mai-1972.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/panama-14-novembre-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/panama-19-septembre-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/papouasie-nouvelle-guinee-20-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-20-juillet-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-4-mai-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-17-septembre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-5-juin-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-26-juillet-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-3-novembre-1978.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-20-novembre-1969.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/perou-27-septembre-1968.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/philippines-19-juillet-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/philippines-20-juin-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/philippines-26-mai-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/philippines-22-janvier-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/philippines-20-decembre-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pologne-21-avril-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pologne-16-fevrier-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pologne-16-decembre-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pologne-19-novembre-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pologne-15-juillet-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/pologne-27-avril-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-26-avril-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-15-septembre-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-24-decembre-2007.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-20-avril-2007.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-25-septembre-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-12-avril-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-15-juin-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-14-decembre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-22-novembre-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-8-juillet-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-centrafricaine-12-juin-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-27-juillet-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-17-novembre-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-25-fevrier-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-22-novembre-2003.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-13-septembre-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-23-juin-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-18-mai-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-15-mai-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-18-septembre-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-20-decembre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-9-juillet-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-11-decembre-1979.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-1-decembre-1977.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rdc-16-juin-1976.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-dominicaine-21-octobre-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-dominicaine-16-avril-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-dominicaine-22-novembre-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/republique-dominicaine-21-mai-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/roumanie-18-mai-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/roumanie-28-juillet-1982.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rwanda-10-mai-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rwanda-7-mars-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/rwanda-21-juillet-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/st-christophe-et-nieves-24-mai-2012.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/st-vincent-&-les-grenadines-18-mars-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sainte-lucie-25-novembre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/samoa-27-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sao-tome-et-principe-12-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sao-tome-et-principe-24-mai-2007.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sao-tome-et-principe-13-septembre-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sao-tome-et-principe-16-mai-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-21-juillet-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-9-juin-2004.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-24-octobre-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-17-juin-1998.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-20-avril-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-3-mars-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-21-juin-1991.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-12-fevrier-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-24-janvier-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-17-novembre-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-21-novembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-18-janvier-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-21-decembre-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-29-novembre-1982.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/senegal-13-octobre-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/serbie-16-novembre-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/seychelles-16-avril-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-14-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-24-janvier-2007.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-10-juillet-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-16-octobre-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-28-mars-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-20-juillet-1994.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-20-novembre-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-19-novembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-8-fevrier-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-8-fevrier-1980.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sierra-leone-15-septembre-1977.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/somalie-13-mars-2024.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/somalie-31-mars-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/somalie-22-juillet-1987.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/somalie-6-mars-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/soudan-15-juillet-2021.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/soudan-3-mai-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/soudan-4-fevrier-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/soudan-18-mars-1982.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/soudan-13-novembre-1979.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sri-lanka-26-juin-2024.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sri-lanka-10-mai-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/suriname-8-octobre-2024.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/suriname-22-juin-2022.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tadjikistan-3-septembre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-23-octobre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-17-janvier-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-14-avril-2000.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-21-janvier-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-21-janvier-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-16-mars-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-13-decembre-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tanzanie-18-septembre-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-20-octobre-2022.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-9-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-24-juin-2015.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-12-juin-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-14-juin-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-28-fevrier-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-24-octobre-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-15-juin-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-16-decembre-2010.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-22-janvier-2009.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-12-juin-2008.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-23-fevrier-1995.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-19-juin-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-9-juillet-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-20-juin-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-22-mars-1988.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-24-juin-1985.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-6-juin-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-12-avril-1983.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-20-fevrier-1981.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/togo-15-juin-1979.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/trinite-&-tobago-27-avril-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/trinite-&-tobago-25-janvier-1989.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/turquie-23-juillet-1980.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/turquie-25-juillet-1979.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/turquie-20-mai-1978.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ukraine-13-juillet-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/vietnam-14-decembre-1993.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/yemen-7-octobre-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/yemen-14-juin-2001.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/yemen-20-novembre-1997.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/yemen-24-septembre-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-13-octobre-2023.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-10-aout-2020.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-11-mai-2005.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-13-septembre-2002.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-16-avril-1999.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-28-fevrier-1996.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-23-juillet-1992.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-12-juillet-1990.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-4-mars-1986.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-20-juillet-1984.html",
"https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-16-mai-1983.html"
]

In [4]:
#@markdown extract_soup(url, queries)

BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://clubdeparis.org/",
    "Connection": "keep-alive",
}

def extract_soup(url, retries=3, timeout=20):
    for attempt in range(retries):
        try:
            r = requests.get(
                    url,
                    headers=BASE_HEADERS,
                    timeout=timeout)#, verify=False)

            # Hard HTTP failure
            if r.status_code >= 500:
                print(r.status_code, url)
                time.sleep(random.uniform(1.5, 3.5))
                continue

            # Soft block (fake 500 page)
            if "error-box" in r.text or "<meta content=\"500\"" in r.text:
                time.sleep(random.uniform(2, 5))
                continue

            return {
                "ok": True,
                "status": r.status_code,
                "url": url,
                "soup": BeautifulSoup(r.text, "html.parser"),
            }

        except Exception as e:
            time.sleep(random.uniform(1.5, 3.5))

    return {
        "ok": False,
        "status": None,
        "url": url,
        "soup": None,
        "error": "blocked_or_server_error"
    }


In [5]:

#@markdown render_block_with_links_at_end(p, BASE_URL = "https://clubdeparis.org")

def render_block_with_links_at_end(node, base_url="https://clubdeparis.org"):
    text_parts = []
    links = []

    for child in node.descendants:
        if isinstance(child, NavigableString):
            text_parts.append(str(child))
        elif isinstance(child, Tag) and child.name == "a":
            href = child.get("href")
            if href:
                if href.startswith("/"):
                    href = urljoin(base_url, href)
                links.append(href)

    text = " ".join(" ".join(text_parts).split())

    if links:
        return text + " " + " ".join(f"<{u}>" for u in links)

    return text


In [6]:
#@markdown extract_paris_club_info(soup, field_map)
def extract_paris_club_info(result, field_map):
    if not result["ok"] or result["soup"] is None:
        return {k: None for k in field_map}
    else:
        soup = result["soup"]
        return extract_paris_club_info_internal(soup, field_map)


In [7]:
#@markdown extract_paris_club_info(soup, field_map)
def extract_paris_club_info_internal(soup, field_map, base_url = "https://clubdeparis.org"):
    data = {}

    # Extract all box2 fields once
    box2 = extract_box2_fields(soup)

    for field, spec in field_map.items():

        # ---- box2 fields (text + links + newlines preserved) ----
        if spec == "box2":
            data[field] = box2.get(field)

        # ---- simple selector ----
        elif isinstance(spec, str):
            el = soup.select_one(spec)
            data[field] = el.get_text(" ", strip=True) if el else None

        # ---- list fields ----
        elif isinstance(spec, dict) and spec["type"] == "list":
            container = soup.select_one(spec["selector"])
            if container:
                data[field] = [li.get_text(" ", strip=True) for li in container.select("li")]
            else:
                data[field] = []

        # ---- attached files ----
        elif isinstance(spec, dict) and spec["type"] == "files":
            links = soup.select(spec["selector"])
            data[field] = [
                f"{a.get_text(strip=True)}<{urljoin(base_url, a['href'])}>"
                for a in links if a.has_attr("href")
            ]

    return data


In [11]:
def extract_box2_fields(soup):
    fields = {}

    for box in soup.select("div.box2"):
        header = box.select_one("h3.box2-header")
        body = box.select_one("div.box2-text")

        if not header or not body:
            continue

        key = header.get_text(" ", strip=True)

        # 🔒 REMOVE creditor / observer groups from narrative extraction
        body_clone = body.__copy__()
        for g in body_clone.select(
            ".group-traitment-body-s10-1, "
            ".group-traitment-body-s10-2, "
            ".group-traitment-body-s10-3"
        ):
            g.decompose()

        blocks = body_clone.find_all(["p", "li"], recursive=True)

        if blocks:
            value = "\n".join(render_block_with_links_at_end(b) for b in blocks)
        else:
            value = body_clone.get_text(" ", strip=True)

        fields[key] = value

    return fields

In [8]:
FIELD_MAP = {
    # Top banner
    "Debtor": ".row.traitements a.colored-cta",

    # Main box2 fields (text + paragraphs + links)
    "Total external debt of the country": "box2",
    "Supporting agreements with the international institutions": "box2",
    "Amounts treated": "box2",
    "Accorded treatment": "box2",
    "Categories of debt treatment": "box2",
    "Repayment profile": "box2",
    "Comparability of treatment provision": "box2",
    "Cut-off date": "box2",
    "Organisation of the session": "box2",

    # Creditor & observer lists
    "Participating creditors": {
        "type": "list",
        "selector": ".group-traitment-body-s10-3"
    },

    "Observers1": {
        "type": "list",
        "selector": ".field-name-field-treatment-obs-c"
    },

    "Observers2": {
        "type": "list",
        "selector": ".field-name-field-treatment-obs-ins"
    },

    "Specific provisions": ".field-name-field-treatment-spec-pro",

    "Attached file": {
        "type": "files",
        "selector": "section.genericLinkList a.download-link"
    }
}


In [9]:
print(list(FIELD_MAP.keys()))

['Debtor', 'Total external debt of the country', 'Supporting agreements with the international institutions', 'Amounts treated', 'Accorded treatment', 'Categories of debt treatment', 'Repayment profile', 'Comparability of treatment provision', 'Cut-off date', 'Organisation of the session', 'Participating creditors', 'Observers1', 'Observers2', 'Specific provisions', 'Attached file']


In [12]:
#@markdown TESTING GROUND

url = "https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/suriname-8-octobre-2024.html"

print('test...')
page = extract_soup(url)
# print(res['soup'])


found = extract_paris_club_info(page, FIELD_MAP)
found



test...


{'Debtor': 'SURINAME',
 'Total external debt of the country': 'US$ 2,694 million of which US$ 88 million due to the Paris Club as of December 31, 2023 (source: IMF)',
 'Supporting agreements with the international institutions': 'IMF programme supported by the arrangements under the Extended Fund Facility (EFF) approved on December 22, 2021 <https://www.imf.org/en/News/Articles/2021/12/22/pr21400-imf-executive-board-approves-extended-arrangement-under-the-extended-fund-facility-suriname>\nDownload the IMF report EFF document <https://www.imf.org/en/Publications/CR/Issues/2021/12/23/Suriname-Request-for-an-Extended-Arrangement-under-the-Extended-Fund-Facility-Press-Release-511294>',
 'Amounts treated': None,
 'Accorded treatment': 'Amendment on the entry into force of the second phase of the 2022 restructuring agreement of its external public debt, thanks to Suriname’s economic performance',
 'Categories of debt treatment': None,
 'Repayment profile': 'Treatment under classic terms <htt

In [14]:
#@markdown prepare empty dataframe and rip information from html

# if this error comes up, please wait for the spreadsheet to refresh
# then run the "Connect to Gsheet" cell again
# MissingSchema: Invalid URL 'Loading...': No scheme supplied. Perhaps you meant https://Loading...?


df = pd.DataFrame(
            columns = FIELD_MAP.keys()
        )

for url in urls:
    if url:
        print(url)
        soup = extract_soup(url)
        fields = extract_paris_club_info(soup, FIELD_MAP)
        new = pd.DataFrame.from_dict(fields, orient='index').T
        new['url'] = url
        df = pd.concat([df, new.set_index('url')])
        time.sleep(random.uniform(1.2, 3.2))
    elif not url:
        pass


https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-17-mars-2010.html
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-18-juillet-2007.html 
500 https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-18-juillet-2007.html 
500 https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-18-juillet-2007.html 
500 https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-18-juillet-2007.html 
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-19-juillet-2006.html
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-

In [15]:
#reorder columns to make sense with previous versions
cols = ["Debtor", "Accorded treatment", "Amounts treated", "Categories of debt treatment",
        "Comparability of treatment provision", "Cut-off date", "Participating creditors",
        "Repayment profile", "Observers1", "Observers2", "Specific provisions",
        "Total external debt of the country",
        "Supporting agreements with the international institutions",
        "Organisation of the session", "Attached file"]
df = df[cols]
df

,Debtor,Accorded treatment,Amounts treated,Categories of debt treatment,Comparability of treatment provision,Cut-off date,Participating creditors,Repayment profile,Observers1,Observers2,Specific provisions,Total external debt of the country,Supporting agreements with the international institutions,Organisation of the session,Attached file
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-17-mars-2010.html,AFGHANISTAN,Debt cancellation following Afghanistan’s havi...,"$1,027 million of which $442 million being can...","Treatment of stock as of January 1, 2010",The Islamic Republic of Afghanistan was declar...,"June 20, 1999","[GERMANY, RUSSIAN FEDERATION, UNITED STATES OF...",Treatment under HIPC Initiative Exit terms\nPa...,"[AUSTRALIA, CANADA, DENMARK, FRANCE, JAPAN, NE...","[IMF, UNCTAD, World Bank]",None,"$2,104 million as of March 20, 2009\n$1,027 mi...",IMF programme supported by an Arrangement unde...,"The meeting was chaired by Mr. Rémy RIOUX, Vic...",[Afghanistan-Cancellation debt-17 03 2010(PDF ...
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-18-juillet-2007.html,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/afghanistan-19-juillet-2006.html,AFGHANISTAN,Significant reduction of its external debt fol...,None,None,In order to secure comparable treatment of its...,"June 20, 1999","[GERMANY, RUSSIAN FEDERATION, UNITED STATES OF...",Treatment under Naples terms (cancellation rat...,"[CANADA, DENMARK, FRANCE, ITALY, JAPAN, NETHER...","[IMF, OECD, UNCTAD, World Bank]",None,"$2,362 million of which being due to Paris Clu...",Programme with the IMF: Arrangement under the ...,The meeting was chaired by Mr. Ambroise Fayoll...,[Afghanistan-Substantial debt relief-19 07 200...
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/albanie-18-janvier-2000.html,ALBANIA,None,$89 million,None,yes,"September 30, 1993","[AUSTRIA, FRANCE, GERMANY, ITALY, JAPAN, NETHE...",Treatment under Classic terms <https://clubdep...,[],[],None,None,None,,[]
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/albanie-22-juillet-1998.html,ALBANIA,None,$75 million,None,yes,"September 30, 1993","[ITALY, RUSSIAN FEDERATION]",Treatment under Naples terms (cancellation rat...,"[AUSTRALIA, AUSTRIA, BELGIUM, CANADA, DENMARK,...",[],None,None,None,Have attended:,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-23-juillet-1992.html,ZAMBIA,None,$918 million,None,None,None,"[AUSTRIA, BELGIUM, BRAZIL, CANADA, FRANCE, GER...",Treatment under London terms (cancellation rat...,"[FINLAND, ISRAEL, NETHERLANDS, NORWAY]","[IMF, UNCTAD, World Bank]",None,None,None,Have attended:,[]
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-12-juillet-1990.html,ZAMBIA,None,$963 million,None,yes,None,"[AUSTRIA, BELGIUM, BRAZIL, CANADA, FRANCE, GER...",Treatment under Toronto terms (cancellation ra...,"[DENMARK, FINLAND, NORWAY, SWEDEN]","[European Commission, IMF, OECD, UNCTAD, World...",None,None,None,Have attended:,[]
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-4-mars-1986.html,ZAMBIA,None,$547 million,None,yes,None,"[AUSTRIA, BELGIUM, BRAZIL, CANADA, FRANCE, GER...",Treatment under Classic terms <https://clubdep...,"[FINLAND, SPAIN]","[European Commission, IMF, OECD, UNCTAD, World...",None,None,None,Have attended:,[]
https://clubdeparis.org/en/sites/clubd

In [16]:
datetime.datetime.now().strftime("%Y-%m-%d_%Hh%Mm%Ss")
nowstring = datetime.datetime.now().strftime("%Y-%m-%d_%Hh%Mm%Ss")
df.to_csv(f"{nowstring}_OpenParisClub.csv")

In [ ]:
 df

,Debtor,Accorded treatment,Amounts treated,Categories of debt treatment,Comparability of treatment provision,Cut-off date,Participating creditors,Repayment profile,Observers1,Observers2,Specific provisions,Total external debt of the country,Supporting agreements with the international institutions,Organisation of the session,Attached file
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/ghana-11-juin-2024.html,GHANA,"For the OCC, a flow treatment is applied, with...",None,None,In order to secure comparable treatment of its...,"December 31, 2022",[],Maturities falling due in a specific year N ar...,[],[],None,"$30.6 billion as of 31 December 2022, includin...",IMF programme supported by the arrangements un...,Key dates:\n- Staff-Level Agreement: 12 Decemb...,[]
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/sri-lanka-26-juin-2024.html,SRI LANKA,"Stock treatment due as of January 1, 2023 resc...",None,None,In order to secure comparable treatment of deb...,"March 18, 2022",[],Stock treatment rescheduled until 2042,[],[],None,"$41.5 billion as of December 31, 2022",IMF programme supported by the arrangement und...,Key dates:\n- Staff-Level Agreement: September...,[]
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/suriname-8-octobre-2024.html,SURINAME,Amendment on the entry into force of the secon...,None,None,In order to secure comparable treatment of deb...,"April 29, 2021","[FRANCE, ISRAEL, ITALY, NETHERLANDS]",Treatment under classic terms <https://clubdep...,"[DENMARK, GERMANY, IRELAND, JAPAN, SPAIN, SWIT...","[IMF, World Bank]",None,"US$ 2,694 million of which US$ 88 million due ...",IMF programme supported by the arrangements un...,"The meeting was chaired by Mr. William ROOS, c...",[08 10 2024-Suriname-Entry into force of secon...
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/tchad-20-octobre-2022.html,CHAD,"After collective assessment, OCC members agree...",None,None,If OCC Members agree to a debt treatment in ac...,NA,[],None,[],[],None,"$3.0 billion as of December 31, 2020, includin...",IMF programme supported by the arrangements un...,Key dates:\n- Staff-Level Agreement: January 2...,[]
https://clubdeparis.org/en/sites/clubdeparis/accueil/accords-signes-avec-le-club-de-p/recherche-accords-signes/fiches/zambie-13-octobre-2023.html,ZAMBIA,"For the OCC, a stock treatment is applied, wit...",None,None,In order to secure comparable treatment of its...,"March 24, 2020",[],Rescheduling of the debt stock until 2043 unde...,[],[],None,"$20.9 billion as of December 31, 2022, includi...",IMF programme supported by the arrangements un...,Key dates:\n- Staff-Level Agreement: December ...,[]


# archive downloadable links

dd 20260121

In [ ]:
#@markdown urls_archive

urls_archive = ["https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Afghanistan-17%20mars%202010/Afghanistan%2017%2003%202010-Cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Afghanistan-17%20mars%202010/Afghanistan%2017%2003%202010-Rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/B%c3%a9nin-23%20avril%202003/Benin-23%2004%202003-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/B%c3%a9nin-24%20octobre%202000/Benin-24%2010%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/B%c3%a9nin-24%20octobre%202000/Benin-24%2010%202000-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Bolivie-10%20juillet%202001/Bolivia-10%2007%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Bolivie-10%20juillet%202001/Bolivia-10%2007%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burkina%20Faso-20%20juin%202002/Burkina%20Faso-20%2006%202002%20and%2013%2003%202003-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burkina%20Faso-20%20juin%202002/Burkina%20Faso-20%2006%202002%20and%2013%2003%202003-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burkina%20Faso-20%20juin%202002/Burkina%20Faso-20%2006%202002%20and%2013%2003%202003-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burkina%20Faso-24%20octobre%202000/Burkina%20Faso-24%2010%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burkina%20Faso-24%20octobre%202000/Burkina%20Faso-24%2010%202000-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burundi-11%20mars%202009/Burundi%2011%2003%2009-Cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burundi-11%20mars%202009/Burundi%2011%2003%2009-Rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burundi-4%20mars%202004/Burundi-04%2003%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burundi-4%20mars%202004/Burundi-04%2003%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Burundi-4%20mars%202004/Burundi-04%2003%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Cameroun-17%20juin%202006/Cameroon-17%2006%202006-Treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Cameroun-17%20juin%202006/Cameroon-17%2006%202006-Cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Cameroun-24%20janvier%202001/Cameroon-24%2001%202001-Treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Cameroun-24%20janvier%202001/Cameroon-24%2001%202001-Cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Cameroun-24%20janvier%202001/Cameroon-24%2001%202001-Rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Comores-28%20f%c3%a9vrier%202013/Comoros-28%2002%202013-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Comores-28%20f%c3%a9vrier%202013/Comoros-28%2002%202013-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Comores-28%20f%c3%a9vrier%202013/Comoros-28%2002%202013-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Comores-19%20novembre%202009/Comores-19%2011%202009-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Comores-19%20novembre%202009/Comores-19%2011%202009-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Comores-19%20novembre%202009/Comores-19%2011%202009-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-29%20juin%202012/Cote%20d'Ivoire-29%2006%202012-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-29%20juin%202012/Cote%20d'Ivoire-29%2006%202012-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-29%20juin%202012/Cote%20d'Ivoire-29%2006%202012-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-15%20novembre%202011/Cote%20d'Ivoire-15%2011%202011-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-15%20novembre%202011/Cote%20d'Ivoire-15%2011%202011-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-15%20novembre%202011/Cote%20d'Ivoire-15%2011%202011-Amounts%20rescheduled%20%26%20deferred.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-15%20mai%202009/Cote%20d'Ivoire-15%2005%202009-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-15%20mai%202009/Cote%20d'Ivoire-15%2005%202009-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-15%20mai%202009/Cote%20d'Ivoire-15%2005%202009-Amounts%20rescheduled%20%26%20deferred.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-10%20avril%202002/Cote%20d'Ivoire-10%2004%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-10%20avril%202002/Cote%20d'Ivoire-10%2004%202002-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/C%c3%b4te%20d'Ivoire-10%20avril%202002/Cote%20d'Ivoire-10%2004%202002-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Djibouti-16%20octobre%202008/Djibouti-16%2010%202008-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Equateur-15%20septembre%202000/Ecuador-15%2009%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ethiopie-13%20mai%202004/Ethiopia-13%2005%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ethiopie-13%20mai%202004/Ethiopia-13%2005%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ethiopie-13%20mai%202004/Ethiopia-13%2005%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ethiopie-5%20avril%202001/Ethiopia-05%2004%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ethiopie-5%20avril%202001/Ethiopia-05%2004%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ethiopie-5%20avril%202001/Ethiopia-05%2004%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/F%c3%a9d%c3%a9ration%20de%20Russie-1%20ao%c3%bbt%201999/Russian%20Federation-01%2008%201999-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Gabon-11%20juin%202004/Gabon-11%2006%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Gabon-15%20d%c3%a9cembre%202000/Gabon-15%2012%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Gambie-24%20janvier%202008/Gambia-24%2001%202008-Amounts%20treated.PDF", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Gambie-24%20janvier%202008/Gambia-24%2001%202008-Amounts%20cancelled.PDF", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Gambie-24%20janvier%202008/Gambia-24%2001%202008-Amounts%20rescheduled.PDF", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Georgie-21%20juillet%202004/Georgia-21%2007%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Georgie-6%20mars%202001/Georgia-06%2003%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ghana-22%20juillet%202004/Ghana-22%2007%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ghana-22%20juillet%202004/Ghana-22%2007%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ghana-16%20mai%202002/Ghana-16%2005%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ghana-16%20mai%202002/Ghana-16%2005%202002-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Ghana-16%20mai%202002/Ghana-16%2005%202002-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Grenade-19%20novembre%202015/Grenade-19%2011%202015-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Grenade-19%20novembre%202015/Grenade-19%2011%202015-Amounts%20rescheduled%20and%20deferred.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Grenade-12%20mai%202006/Grenade-12%2005%202006-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-25%20octobre%202012/Guinea-25%2010%202012-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-25%20octobre%202012/Guinea-25%2010%202012-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-25%20octobre%202012/Guinea-25%2010%202012-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-11%20avril%202012/Guinea-11%2004%202012-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-11%20avril%202012/Guinea-11%2004%202012-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-11%20avril%202012/Guinea-11%2004%202012-Amounts%20rescheduled%20%26%20deferred.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-23%20janvier%202008/Guinea-23%2001%202008-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-23%20janvier%202008/Guinea-23%2001%202008-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-23%20janvier%202008/Guinea-23%2001%202008-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-15%20mai%202001/Guinea-15%2005%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-15%20mai%202001/Guinea-15%2005%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e-15%20mai%202001/Guinea-15%2005%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20Bissau-10%20mai%202011/Guinea%20Bissau-10%2005%202011-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20Bissau-10%20mai%202011/Guinea%20Bissau-10%2005%202011-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20Bissau-10%20mai%202011/Guinea%20Bissau-10%2005%202011-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20Bissau-6%20juillet%202010/Guine%20Bissau-06%2007%202010-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20Bissau-6%20juillet%202010/Guine%20Bissau-06%2007%202010-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20Bissau-6%20juillet%202010/Guine%20Bissau-06%2007%202010-Amounts%20rescheduled%20%26%20deferred.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20BIssau-26%20janvier%202001/Guinea%20Bissau-26%2001%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20BIssau-26%20janvier%202001/Guinea%20Bissau-26%2001%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guin%c3%a9e%20BIssau-26%20janvier%202001/Guinea%20Bissau-26%2001%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guyana-14%20janvier%202004/Guyana-14%2001%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/1-ACCORDS%20CONCLUS%20-%20Afghanistan%20%c3%a0%20Guyana/Guyana-14%20janvier%202004/Guyana-14%2001%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Haiti-8%20juillet%202009/Haiti-08%2007%202009-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Haiti-8%20juillet%202009/Haiti-08%2007%202009-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Haiti-8%20juillet%202009/Haiti-08%2007%202009-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Haiti-12%20d%c3%a9cembre%202006/Haiti-12%2012%202006-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Haiti-12%20d%c3%a9cembre%202006/Haiti-12%2012%202006-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Haiti-12%20d%c3%a9cembre%202006/Haiti-12%2012%202006-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Honduras-12%20mai%202005/Honduras-12%2005%202005-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Honduras-12%20mai%202005/Honduras-12%2005%202005-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Honduras-12%20mai%202005/Honduras-12%2005%202005-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Honduras-14%20avril%202004/Honduras-14%2004%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Honduras-14%20avril%202004/Honduras-14%2004%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Honduras-14%20avril%202004/Honduras-14%2004%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Indon%c3%a9sie-10%20mai%202005/Indonesia-10%2005%202005-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Indon%c3%a9sie-12%20avril%202002/Indonesia-12%2004%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Irak-21%20novembre%202004/Iraq-21%2011%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Irak-21%20novembre%202004/Iraq-21%2011%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Irak-21%20novembre%202004/Iraq-21%2011%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Jordanie-10%20juillet%202002/Jordan-10%2007%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Jordanie-20%20mai%201999/Jordan-20%2005%201999-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Kenya-15%20janvier%202004/Kenya-15%2001%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Kirghizie-11%2003%202005/Kyrgyz%20Republic-11%2003%202005-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Kirghizie-11%2003%202005/Kyrgyz%20Republic-11%2003%202005-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Kirghizie-11%2003%202005/Kyrgyz%20Republic-11%2003%202005-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Kirghizie-7%20mars%202002/Kirgyzstan-07%2003%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Liberia-16%20septembre%202010/Liberia-16%2009%202010-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Liberia-16%20septembre%202010/Liberia-16%2009%202010-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Liberia-16%20septembre%202010/Liberia-16%2009%202010-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Liberia-17%20avril%202008/Liberia-17%2004%202008-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Liberia-17%20avril%202008/Liberia-17%2004%202008-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Madagascar-16%20novembre%202004/Madagascar-16%2011%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Madagascar-16%20novembre%202004/Madagascar-16%2011%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Madagascar-16%20novembre%202004/Madagascar-16%2011%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Madagascar-7%20mars%202001/Madagascar-07%2003%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Madagascar-7%20mars%202001/Madagascar-07%2003%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Madagascar-7%20mars%202001/Madagascar-07%2003%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Malawi-19%20octobre%202006/Malawi-19%2010%202006-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Malawi-19%20octobre%202006/Malawi-19%2010%202006-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Malawi-25%20janvier%202001/Malawi-25%2001%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Malawi-25%20janvier%202001/Malawi-25%2001%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Malawi-25%20janvier%202001/Malawi-25%2001%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mali-12%20mars%202003/Mali-12%2003%202003-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mali-12%20mars%202003/Mali-12%2003%202003-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mali-25%20octobre%202000/Mali-%2025%2010%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mali-25%20octobre%202000/Mali-%2025%2010%202000-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mauritanie-8%20juillet%202002/Mauritania-08%2007%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mauritanie-8%20juillet%202002/Mauritania-08%2007%202002-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Mauritanie-8%20juillet%202002/Mauritania-08%2007%202002-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/2-ACCORDS%20CONCLUS-Haiti%20%c3%a0%20Mozambique/Moldavie-12%20mai%202006/Moldova-12%2005%202006-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Myanmar-25%20janvier%202013/Myanmar-25%2001%202013-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Myanmar-25%20janvier%202013/Myanmar-25%2001%202013-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Myanmar-25%20janvier%202013/Myanmar-25%2001%202013--Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nicaragua-4%20mars%202004/Nicaragua-04%2003%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nicaragua-4%20mars%202004/Nicaragua-04%2003%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nicaragua-4%20mars%202004/Nicaragua-04%2003%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nicaragua-13%20d%c3%a9cembre%202002/Nicaragua-13%2012%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nicaragua-13%20d%c3%a9cembre%202002/Nicaragua-13%2012%202002-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nicaragua-13%20d%c3%a9cembre%202002/Nicaragua-13%2012%202002-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Niger-12%20mai%202004/Niger-12%2005%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Niger-12%20mai%202004/Niger-12%2005%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Niger-25%20janvier%202001/Niger-25%2001%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Niger-25%20janvier%202001/Niger-25%2001%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Nigeria-13%20d%c3%a9cembre%202000/Nigeria-13%2012%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Pakistan-13%20d%c3%a9cembre%202001/Pakistan-13%2012%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-17%20novembre%202010/DRC-17%2011%202010-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-17%20novembre%202010/DRC-17%2011%202010-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-25%20f%c3%a9vrier%202010/DRC-25%2002%202010-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-25%20f%c3%a9vrier%202010/DRC-25%2002%202010-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-25%20f%c3%a9vrier%202010/DRC-25%2002%202010-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-13%20septembre%202002/DRC-13%2009%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/RDC-13%20septembre%202002/DRC-13%2009%202002-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Rwanda-10%20mai%202005/Rwanda-10%2005%202005-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Rwanda-10%20mai%202005/Rwanda-10%2005%202005-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Rwanda-10%20mai%202005/Rwanda-10%2005%202005-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Sao%20Tome%20et%20Principe-24%20mai%202007/Sao%20Tom%c3%a9%20%26%20Principe-24%2005%202007-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/3-ACCORDS%20CONCLUS-Myanmar%20%c3%a0%20Sao%20Tom%c3%a9/Sao%20Tome%20et%20Principe-24%20mai%202007/Sao%20Tom%c3%a9%20%26%20Principe-24%2005%202007-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/S%c3%a9n%c3%a9gal-9%20juin%202004/Senegal-09%2006%202004-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/S%c3%a9n%c3%a9gal-9%20juin%202004/Senegal-09%2006%202004-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/S%c3%a9n%c3%a9gal-9%20juin%202004/Senegal-09%2006%202004-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/S%c3%a9n%c3%a9gal-24%20octobre%202000/Senegal-24%2010%202000-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/S%c3%a9n%c3%a9gal-24%20octobre%202000/Senegal-24%2010%202000--Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sierra%20Leone-24%20janvier%202007/Sierra%20Leone-24%2001%202007-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sierra%20Leone-24%20janvier%202007/Sierra%20Leone-24%2001%202007-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sierra%20Leone-24%20janvier%202007/Sierra%20Leone-24%2001%202007-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sierra%20Leone-16%20octobre%202001/Sierra%20Lone-16%2010%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sierra%20Leone-16%20octobre%202001/Sierra%20Lone-16%2010%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sierra%20Leone-16%20octobre%202001/Sierra%20Lone-16%2010%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Somalia-13%20mars%202024/Somalia-13%2003%202024-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Somalia-13%20mars%202024/Somalia-13%2003%202024-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Somalia-13%20mars%202024/Somalia-13%2003%2020424-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Somalie-31%20mars%202020/Somalia-31%2003%202020-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Somalie-31%20mars%202020/Somalia-31%2003%202020-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Somalie-31%20mars%202020/Somalia-31%2003%202020-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Soudan-15%20juillet%202021/Sudan-15%2007%202021-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Soudan-15%20juillet%202021/Sudan-15%2007%202021-Amounts%20cancelled.pdf", "https://clubdeparis.org/sites/default/files/sudan-15_07_2021-amounts_rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Sri%20Lanka-10%20mai%202005/Sri%20Lanka-10%2005%202005-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Suriname-22%20juin%202022/Suriname-22%2006%202022-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Suriname-22%20juin%202022/Suriname-22%2006%202022-Amounts%20rescheduled%20%26%20deferred.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tanzanie-17%20janvier%202002/Tanzania-17%2001%202002-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tanzanie-17%20janvier%202002/Tanzania-17%2001%202002-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tanzanie-17%20janvier%202002/Tanzania-17%2001%202002-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tchad-24%20juin%202015/Chad-24%2006%202015-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tchad-24%20juin%202015/Chad-24%2006%202015-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tchad-24%20juin%202015/Chad-24%2006%202015-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tchad-12%20juin%202001/Chad-13%2006%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tchad-12%20juin%202001/Chad-13%2006%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Tchad-12%20juin%202001/Chad-13%2006%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Togo-16%20d%c3%a9cembre%202010/Togo-16%2012%202010-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Togo-16%20d%c3%a9cembre%202010/Togo-16%2012%202010-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Togo-16%20d%c3%a9cembre%202010/Togo-16%2012%202010--Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Togo-12%20juin%202008/Togo-12%2006%202008-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Togo-12%20juin%202008/Togo-12%2006%202008-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Ukraine-13%20juillet%202001/Ukraine-13%2007%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Y%c3%a9men-14%20juin%202001/Yemen-14%2006%202001-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Y%c3%a9men-14%20juin%202001/Yemen-14%2006%202001-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Y%c3%a9men-14%20juin%202001/Yemen-14%2006%202001-Amounts%20rescheduled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Y%c3%a9men-11%20mai%202005/Zambia-11%2005%202005-Amounts%20treated.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Y%c3%a9men-11%20mai%202005/Zambia-11%2005%202005-Amounts%20cancelled.pdf", "https://clubdeparis.org/files/live/sites/clubdeparis/files/contributed/01%20Fiches%20de%20traitement-PJ/4-ACCORDS%20CONCLUS-S%c3%a9n%c3%a9gal%20%c3%a0%20Zambie/Y%c3%a9men-11%20mai%202005/Zambia-11%2005%202005-Amounts%20rescheduled.pdf"]

In [ ]:
archive_dir = projectpath + 'url_archive/'
if not os.path.exists(archive_dir):
    os.makedirs(archive_dir)

for url in urls_archive:
    #download pdf using requests
    filename = " - ".join(urllib.parse.unquote(url).split('/')[-3:])

    #check if file already exists
    if os.path.exists(archive_dir + filename):
        print("Already exists:",archive_dir + filename)
    else:
        response = requests.get(url)
        with open(archive_dir + filename, 'wb') as f:
            f.write(response.content)
            print(filename)

Already exists: url_archive/1-ACCORDS CONCLUS - Afghanistan à Guyana - Afghanistan-17 mars 2010 - Afghanistan 17 03 2010-Cancelled.pdf
Already exists: url_archive/1-ACCORDS CONCLUS - Afghanistan à Guyana - Afghanistan-17 mars 2010 - Afghanistan 17 03 2010-Rescheduled.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Bénin-23 avril 2003 - Benin-23 04 2003-Amounts treated.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Bénin-24 octobre 2000 - Benin-24 10 2000-Amounts treated.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Bénin-24 octobre 2000 - Benin-24 10 2000-Amounts cancelled.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Bolivie-10 juillet 2001 - Bolivia-10 07 2001-Amounts treated.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Bolivie-10 juillet 2001 - Bolivia-10 07 2001-Amounts cancelled.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Burkina Faso-20 juin 2002 - Burkina Faso-20 06 2002 and 13 03 2003-Amounts treated.pdf
1-ACCORDS CONCLUS - Afghanistan à Guyana - Burkina Faso-20 juin 2002